### Exercise 4: Implement Factory Pattern for Data Transformations
#### Objective:
• Practice Factory Pattern for scalable creation of transformation objects.

• Apply abstraction and composition.
#### Concept:
Instead of manually creating objects, use a factory that decides what transformation to apply.
#### Instructions:
1. Create an abstract class DataTransform with a method apply(df).
2. Implement subclasses:

- NormalizeColumns

- RemoveDuplicates

- StandardizeText

3. Create a TransformFactory that returns transformation objects based on string input.

In [1]:
import time
import functools
from abc import ABC, abstractmethod

import pandas as pd

In [2]:
# from excercise 3
def log_action(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        print(f"[LOG] Starting {self.__class__.__name__}.{func.__name__}")
        result = func(self, *args, **kwargs)
        print(f"[LOG] Finished {self.__class__.__name__}.{func.__name__}")
        return result
    return wrapper
 
 
def log_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[TIME] {func.__name__} took {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

In [3]:
# create an abstract class
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

In [4]:
# subclass: NormalizeColumns
class NormalizeColumns(DataTransform):
    """Min-max scales numeric columns to the range [0, 1]."""

    def __init__(self, columns=None):
        self.columns = columns

    @log_action
    @log_time
    def apply(self, df):
        df = df.copy()
        cols = self.columns or df.select_dtypes(include="number").columns.tolist()
        for col in cols:
            col_min, col_max = df[col].min(), df[col].max()
            if col_max == col_min:
                df[col] = 0.0
            else:
                df[col] = (df[col] - col_min) / (col_max - col_min)
        return df

# subclass: RemoveDuplicates
class RemoveDuplicates(DataTransform):
    """Drops duplicate rows (optionally based on a subset of columns)."""
 
    def __init__(self, subset=None):
        self.subset = subset
 
    @log_action
    @log_time
    def apply(self, df):
        before = len(df)
        df = df.drop_duplicates(subset=self.subset).reset_index(drop=True)
        print(f"   -> removed {before - len(df)} duplicate row(s)")
        return df

# subclass: StandardizeText
class StandardizeText(DataTransform):
    """Strips spaces and unifies the case of text columns."""
 
    def __init__(self, columns=None, case="title"):
        self.columns = columns  # None -> all text columns
        self.case = case        # "lower", "upper" or "title"
 
    @log_action
    @log_time
    def apply(self, df):
        df = df.copy()
        cols = self.columns or df.select_dtypes(include="object").columns.tolist()
        for col in cols:
            text = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
            df[col] = getattr(text.str, self.case)()
        return df

In [5]:
# create Factory
class TransformFactory:
    """Returns a transformation object from its name, so client code
    never has to instantiate the concrete classes itself."""
 
    _registry = {
        "normalize": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }
 
    @classmethod
    def create(cls, name: str, **kwargs) -> DataTransform:
        key = name.strip().lower()
        if key not in cls._registry:
            raise ValueError(
                f"Unknown transformation '{name}'. Available: {list(cls._registry)}"
            )
        return cls._registry[key](**kwargs)
 
    @classmethod
    def register(cls, name: str, transform_cls):
        """Add a new transformation without modifying the factory (Open/Closed)."""
        if not issubclass(transform_cls, DataTransform):
            raise TypeError("transform_cls must inherit from DataTransform")
        cls._registry[name.lower()] = transform_cls

In [6]:
# ---------------------------------------------------------------
# test
# ---------------------------------------------------------------
if __name__ == "__main__":
    df = pd.read_csv("sample_data (2).csv")
 
    # Add messy text + a duplicate row so every transformation has something to do
    df.loc[len(df)] = df.loc[0]
    df.loc[1, "city"] = "  los   angeles "
    print("Original data:\n", df, "\n")
 
    steps = [
        ("remove_duplicates", {}),
        ("standardize_text", {"columns": ["name", "city"]}),
        ("normalize", {"columns": ["age", "height_cm", "weight_kg", "score"]}),
    ]
 
    for name, params in steps:
        transform = TransformFactory.create(name, **params)  # the factory decides
        df = transform.apply(df)                             # polymorphism
        print()
 
    print("Transformed data:\n", df.round(3))

Original data:
    id     name   age  height_cm  weight_kg              city  score
0   1    Alice  29.0      165.0       68.0          New York   85.0
1   2      Bob   NaN      172.0        NaN    los   angeles    90.0
2   3  Charlie  35.0      168.0       72.0           Chicago    NaN
3   4    David   NaN        NaN       80.0           Houston   75.0
4   5      Eva  27.0      160.0       55.0          New York   88.0
5   6    Frank  30.0      175.0       85.0       Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0           Chicago   92.0
7   1    Alice  29.0      165.0       68.0          New York   85.0 

[LOG] Starting RemoveDuplicates.apply
   -> removed 1 duplicate row(s)
[TIME] apply took 0.0125s
[LOG] Finished RemoveDuplicates.apply

[LOG] Starting StandardizeText.apply
[TIME] apply took 0.0285s
[LOG] Finished StandardizeText.apply

[LOG] Starting NormalizeColumns.apply
[TIME] apply took 0.0088s
[LOG] Finished NormalizeColumns.apply

Transformed data:
    id     n